# Notebook 01: GWAS to Target Discovery

This tutorial demonstrates how to:
1. Ingest raw Genome-Wide Association Study (GWAS) summary statistics.
2. Perform quality control filtering (p-value, minor allele frequency).
3. Perform distance-based locus clumping to identify lead causal variants.
4. Aggregate single-variant statistics to gene-level evidence using **GenT** (Cauchy combination test).
5. Visualize the genome-wide landscape via a multi-chromosome **Manhattan Plot**.

In [ ]:
import os
import sys
# Add src/ to path
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
from target_discovery.gwas_parser import GWASParser
from target_discovery.gent_wrapper import GenTWrapper
from utils.viz_utils import plot_manhattan

## 1. Load GWAS Summary Statistics

We load benchmark summary statistics containing variants associated with complex metabolic, neurological, and oncogenic traits.

In [ ]:
gwas_path = "../data/sample/example_gwas.tsv"
parser = GWASParser(gwas_path)
gwas_df = parser.load_data()
print(f"Loaded {len(gwas_df)} variants across {gwas_df['CHR'].nunique()} chromosomes.")
gwas_df.head(10)

## 2. Identify Genome-Wide Significant Loci

The standard genome-wide threshold is $p < 5 \times 10^{-8}$ (correcting for ~1,000,000 independent tests in European LD blocks).

In [ ]:
sig_variants = parser.filter_significant_variants(p_threshold=5e-8)
print(f"Found {len(sig_variants)} genome-wide significant variants:")
sig_variants[["SNP", "CHR", "BP", "GENE", "BETA", "PVALUE", "LOG10_P"]]

## 3. Distance-based Clumping to Identify Independent Leads

In [ ]:
lead_snps = parser.get_lead_snps_per_locus(window_bp=500_000, p_threshold=5e-8)
print(f"Identified {len(lead_snps)} independent lead loci:")
lead_snps[["SNP", "CHR", "BP", "GENE", "PVALUE"]]

## 4. Gene-Level Aggregation with GenT (Cauchy Combination Test)

Single SNPs often miss polygenic burden. GenT aggregates multi-SNP associations across the gene body into an analytical gene-level p-value.

In [ ]:
gent = GenTWrapper()
gene_rankings = gent.aggregate_gene_pvalues(gwas_df)
print("Top Prioritized Druggable Target Genes:")
gene_rankings.head(10)

## 5. Visualize Manhattan Plot

In [ ]:
fig = plot_manhattan(gwas_df, p_threshold=5e-8)
plt.show()